# 07. Searchable Transcript

## Goal

Split cleaned transcript rows into searchable chunks and return the chunks most relevant to a question.

## Why This Step Matters

Meeting transcripts can be long. Before summarization, we should retrieve only the transcript parts related to the user's question.

## Inputs

- Cleaned timestamped transcript rows.
- Retrieval utilities from `src/retrieval/`.

## Outputs

- Transcript chunks with time ranges.
- Top-k relevant chunks for a question.
- Optional local JSON results under ignored `outputs/`.

## Setup

This notebook starts with a tiny cleaned transcript sample so retrieval behavior is easy to inspect.

In [8]:
import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.retrieval.chunking import chunk_transcript_rows
from src.retrieval.search import search_chunks
from src.transcript.clean import clean_transcript_rows

SAVE_OUTPUTS = False
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "retrieval"
MAX_CHUNK_CHARACTERS = 90
TOP_K = 2

print("Project root:", PROJECT_ROOT)

Project root: /home/lataeq/Code/Saranote


## Load Data

Use sample transcript rows that look like the output from Phase 7.

In [9]:
raw_rows = [
    {"timestamp": "00:00:00 - 00:00:08", "text": "  เปิดประชุม   และแจ้งวาระวันนี้  "},
    {"timestamp": "00:00:08 - 00:00:18", "text": "ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครงการใหม่"},
    {"timestamp": "00:00:18 - 00:00:28", "text": "ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์"},
    {"timestamp": "00:00:28 - 00:00:38", "text": "มีการพูดถึง ความเสี่ยง เรื่องข้อมูลไม่ครบ"},
    {"timestamp": "00:00:38 - 00:00:45", "text": "   "},
]

cleaned_rows = clean_transcript_rows(raw_rows)
pd.DataFrame(cleaned_rows)

,timestamp,text
0,00:00:00 - 00:00:08,เปิดประชุม และแจ้งวาระวันนี้
1,00:00:08 - 00:00:18,ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครง...
2,00:00:18 - 00:00:28,ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์
3,00:00:28 - 00:00:38,มีการพูดถึง ความเสี่ยง เรื่องข้อมูลไม่ครบ


## Run Experiment

Split the cleaned transcript into chunks that keep start and end timestamps.

In [10]:
chunks = chunk_transcript_rows(cleaned_rows, max_characters=MAX_CHUNK_CHARACTERS)

chunks_df = pd.DataFrame([
    {
        "chunk": chunk.index,
        "start_timestamp": chunk.start_timestamp,
        "end_timestamp": chunk.end_timestamp,
        "row_count": chunk.row_count,
        "text": chunk.text,
    }
    for chunk in chunks
])
chunks_df

,chunk,start_timestamp,end_timestamp,row_count,text
0,1,00:00:00 - 00:00:08,00:00:08 - 00:00:18,2,เปิดประชุม และแจ้งวาระวันนี้\nที่ประชุม มีมติ ...
1,2,00:00:18 - 00:00:28,00:00:28 - 00:00:38,2,ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์\nมีการ...


## Search Relevant Chunks

Ask a question and return the chunks with the strongest keyword match.

In [11]:
question = "ที่ประชุมมีมติเรื่องงบประมาณอย่างไร"
results = search_chunks(question, chunks, top_k=TOP_K)

results_df = pd.DataFrame([
    {
        "rank": rank,
        "score": round(result.score, 3),
        "matched_terms": ", ".join(result.matched_terms),
        "start_timestamp": result.chunk.start_timestamp,
        "end_timestamp": result.chunk.end_timestamp,
        "text": result.chunk.text,
    }
    for rank, result in enumerate(results, start=1)
])

print("Question:", question)
results_df

Question: ที่ประชุมมีมติเรื่องงบประมาณอย่างไร


,rank,score,matched_terms,start_timestamp,end_timestamp,text
0,1,0.531,,00:00:00 - 00:00:08,00:00:08 - 00:00:18,เปิดประชุม และแจ้งวาระวันนี้\nที่ประชุม มีมติ ...
1,2,0.125,,00:00:18 - 00:00:28,00:00:28 - 00:00:38,ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์\nมีการ...


## Search Checks

Run small expected-output checks so the notebook shows whether retrieval is working.

In [12]:
search_cases = [
    {"query": "งบประมาณ", "expected_contains": "งบประมาณ"},
    {"query": "ส่งเอกสาร", "expected_contains": "ส่งเอกสาร"},
    {"query": "ความเสี่ยง", "expected_contains": "ความเสี่ยง"},
    {"query": "ที่ประชุมสรุปเรื่องงบประมาณอย่างไร", "expected_contains": "งบประมาณ"},
]

check_rows = []
for case in search_cases:
    case_results = search_chunks(case["query"], chunks, top_k=1)
    top_text = case_results[0].chunk.text if case_results else ""
    check_rows.append({
        "query": case["query"],
        "expected_contains": case["expected_contains"],
        "top_text": top_text,
        "passed": case["expected_contains"] in top_text,
    })

search_checks_df = pd.DataFrame(check_rows)
print(f"Passed {search_checks_df['passed'].sum()}/{len(search_checks_df)} search checks")
search_checks_df

Passed 4/4 search checks


,query,expected_contains,top_text,passed
0,งบประมาณ,งบประมาณ,เปิดประชุม และแจ้งวาระวันนี้\nที่ประชุม มีมติ ...,True
1,ส่งเอกสาร,ส่งเอกสาร,ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์\nมีการ...,True
2,ความเสี่ยง,ความเสี่ยง,ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์\nมีการ...,True
3,ที่ประชุมสรุปเรื่องงบประมาณอย่างไร,งบประมาณ,เปิดประชุม และแจ้งวาระวันนี้\nที่ประชุม มีมติ ...,True


In [13]:
assert search_checks_df["passed"].all(), "Some searchable transcript checks failed"

retrieval_summary = pd.DataFrame([
    {
        "cleaned_rows": len(cleaned_rows),
        "chunks": len(chunks),
        "question": question,
        "results_returned": len(results),
        "top_score": round(results[0].score, 3) if results else 0,
    }
])

retrieval_summary

,cleaned_rows,chunks,question,results_returned,top_score
0,4,2,ที่ประชุมมีมติเรื่องงบประมาณอย่างไร,2,0.531


## Optional Save

`outputs/` is ignored, so retrieval artifacts stay local by default.

In [14]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_rows = [
        {
            "score": result.score,
            "matched_terms": result.matched_terms,
            "start_timestamp": result.chunk.start_timestamp,
            "end_timestamp": result.chunk.end_timestamp,
            "text": result.chunk.text,
        }
        for result in results
    ]
    (OUTPUT_DIR / "search_results.json").write_text(
        json.dumps(output_rows, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved search results to:", OUTPUT_DIR)